# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/soumyajeetrc/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring** (Lane 2 in `docs/ml-intern-dataset-and-lane-guide.md`). Not freestyle.

I am choosing this lane because the decision is concrete: with limited editor time, **which page should be reviewed first** for refresh, expansion, protection, pruning, or monitoring? The starter dataset is already one row per content item, and the starter notebooks already showed that a learned ranker can beat a hand-written rule on this slice. That is a reason to stay in this lane — not a reason to treat the starter model as finished. The starter label is a same-window proxy (`trend_direction == "down"`), so the next weeks are for a safer contract, a transparent baseline queue, and later a future-window check on the warehouse. I can confirm or change this through Week 4.

In [11]:
import os
import sys
from pathlib import Path

repo_name = "flyrank-internship-ml"
data_file_path_relative = Path("data/raw/content_refresh_anonymized.csv")

# Store the initial current working directory
initial_cwd = Path.cwd()
repo_root = None

# --- Step 1: Try to find the repo root by looking for the data file ---
# This covers cases where the notebook is run from within the repo structure
# (e.g., work/notebooks, or the repo root itself).
for candidate_dir in [initial_cwd, *initial_cwd.parents]:
    if (candidate_dir / data_file_path_relative).exists():
        repo_root = candidate_dir
        break

# --- Step 2: If data not found, try to clone the repo ---
if repo_root is None:
    cloned_repo_dir = initial_cwd / repo_name
    if not cloned_repo_dir.is_dir():
        print(f"Cloning {repo_name} repository to get data...")
        # Use !git clone for Colab environment
        get_ipython().system(f"git clone https://github.com/soumyajeetrc/{repo_name}.git")

    # After cloning (or if it already existed in initial_cwd/repo_name),
    # set the repo_root to the cloned directory if the data file is there.
    if (cloned_repo_dir / data_file_path_relative).exists():
        repo_root = cloned_repo_dir

# --- Step 3: Fallback if repo_root still not found (shouldn't happen with cloning) ---
if repo_root is None:
    print("Warning: Could not find repository root or data file, defaulting to current directory.")
    repo_root = initial_cwd

# Change to the determined repository root
os.chdir(repo_root)

# Check for the CSV path *relative to the new working directory*
csv_path = data_file_path_relative
assert csv_path.exists(), f"starter CSV not found at {csv_path.absolute()} after changing directory to {os.getcwd()}."

print("Working dir:", os.getcwd())
print("Starter CSV:", csv_path)
print("Python:", sys.version.split()[0])
print("Lane for this notebook: Refresh / Content Opportunity Scoring (provisional)")

Working dir: /content/flyrank-internship-ml
Starter CSV: data/raw/content_refresh_anonymized.csv
Python: 3.13.15
Lane for this notebook: Refresh / Content Opportunity Scoring (provisional)


## 2. The question: decision, action, cost of a wrong call

**Search question.** Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

**Unit of analysis (grain).** One **content item / page**. Not a client, not a calendar day. The starter table is one row per `content_id`. Later warehouse daily facts can build features and a later-window label, but the *output* stays a page-level queue.

**Decision this improves.** Given a small review budget (about 20–50 pages), which existing pages go in today's queue? This is ranking / scoring, not "predict decline" for its own sake.

**Who acts, and what they do.** A content or SEO reviewer (or an operator helping them) opens a ranked list with reason codes, inspects the page, and chooses refresh, expand, monitor, or skip. The model does not publish anything.

**Output.** A ranked review queue: pseudonymized page id, score, suggested action, reason codes, confidence. Decision-support for a human.

**Cost of a wrong call.**
- **False positive** (high on the list, nothing useful to do): wasted editor hours and noisy alerts.
- **False negative** (a high-demand weak page buried): continued missed clicks while the team works lower-value pages.
- **Unranked "everything is down":** if more than half the catalog sits in a decline bucket, a dump of all "down" pages is not a decision — it is a backlog.

**Why data or ML can help — and when a rule is enough.** A dashboard shows symptoms. A transparent rule (stale + visible, low CTR at a given position, declining with demand) is the right baseline and may be enough for some slices. Ranking earns its keep if several overlapping signals (volume, position, freshness, CTR, engagement, movement) do not share one obvious threshold, *and* a leakage-safe split shows the ranker beats the rule on **precision@K** (the metric that matches "review the top K"). If it does not beat the rule, we keep the rule. This is not a project whose success is "we trained a model."

**One-paragraph frame.** For a content reviewer deciding which pages to inspect first under limited capacity, we will build a ranked action queue from observable search and content signals, scoring review priority against a later-defined outcome (starter: a same-window decline *proxy*; capstone aim: a future window). Success is precision@K versus a transparent baseline, on a client-grouped (and later time-aware) split. A wrong call wastes editor time or hides a high-demand page. A single if-statement is not enough because volume, position, freshness, and CTR stack in different combinations. We will claim only observed, directional, decision-support results — not that a refresh caused recovery, and not a Google ranking factor.

In [12]:
# Capacity check: an unranked "review every down page" policy is not a decision.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
n = len(df)
n_down = int((df["trend_direction"] == "down").sum())
review_budget = 50

print(f"Pages in starter slice: {n:,}")
print(f"Pages with trend_direction == 'down': {n_down:,} ({n_down / n:.1%})")
print(f"If a reviewer can inspect {review_budget} pages, that is {review_budget / n_down:.2%} of the 'down' bucket.")
print("So the decision is ranking under capacity, not listing every 'down' row.")


Pages in starter slice: 30,000
Pages with trend_direction == 'down': 16,262 (54.2%)
If a reviewer can inspect 50 pages, that is 0.31% of the 'down' bucket.
So the decision is ranking under capacity, not listing every 'down' row.


## 3. Quick look at the data (2-3 real numbers)

The cell below loads `data/raw/content_refresh_anonymized.csv` (30,000 pages, 32 clients, trailing-90-day metrics). Three numbers from that slice are why Lane 2 looks worth the next weeks:

1. **Capacity.** 16,262 pages (54.2%) sit in `trend_direction == "down"`. A reviewer who can check 50 pages cannot work from that flag alone.
2. **Demand is in the messy middle, not in one stale-page rule.** Pages that are declining *and* have `impressions_90d >= 100` are 13,152 rows and carry about **51% of all impressions**. The starter "stale visible page" rule (`days_since_last_update >= 180` and `impressions_90d >= 500`) matches only **17** pages. A single freshness if-statement barely makes a queue.
3. **CTR gaps on already-visible pages.** 9,759 pages have `impressions_90d >= 500`, a recorded position in 1–20, and `ctr < 0.5` (that is 0.5%, not 50% — rates in this file are ×100 percentages). Those pages carry about **58% of impressions**. Reviewing metadata/content on visible under-clickers is a real action, but it still has to be ranked and position-aware.

Together: lots of "down" labels, impression mass on overlapping weakness patterns, and a naive stale rule that almost never fires. That is a ranking problem with a useful baseline, not a reason to skip the baseline and jump to a model.

*(The starter notebooks already reported baseline precision@50 ≈ 0.24 vs random forest ≈ 0.74 on a client-holdout split. That is encouraging on this slice. It used the same-window proxy label, so it is not a capstone result — it is a reason this lane has something to beat.)*

In [13]:
# 2–3 real numbers from the starter snapshot. Aggregates only — no ids, names, or queries.
assert "df" in dir(), "run the cells above first (Run all)"

total_impr = df["impressions_90d"].sum()
n_down = int((df["trend_direction"] == "down").sum())

declining_with_demand = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
stale_visible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)
pos_missing = df["avg_position"] == 0

print("Starter grain: one row per content item")
print(f"  rows={len(df):,}  columns={df.shape[1]}  clients={df['client_id'].nunique()}")
print()
print("1) Decline bucket (same-window proxy, not a future label)")
print(f"   trend_direction=='down': {n_down:,} pages  ({n_down / len(df):.1%})")
print()
print("2) Demand vs a single stale rule")
print(
    f"   declining AND impressions_90d>=100: {int(declining_with_demand.sum()):,} pages, "
    f"{df.loc[declining_with_demand, 'impressions_90d'].sum() / total_impr:.1%} of impressions"
)
print(
    f"   stale visible (update>=180d AND impressions_90d>=500): {int(stale_visible.sum()):,} pages, "
    f"{df.loc[stale_visible, 'impressions_90d'].sum() / total_impr:.1%} of impressions"
)
print()
print("3) Visible low-CTR pages (ctr is a ×100 percentage: 0.5 means 0.5%)")
print(
    f"   impressions_90d>=500, position 1-20, ctr<0.5: {int(low_ctr_visible.sum()):,} pages, "
    f"{df.loc[low_ctr_visible, 'impressions_90d'].sum() / total_impr:.1%} of impressions"
)
print()
print("Data gotcha check (not a lane metric): avg_position==0 means no data, not rank zero")
print(f"   rows with avg_position==0: {int(pos_missing.sum()):,}")


Starter grain: one row per content item
  rows=30,000  columns=44  clients=32

1) Decline bucket (same-window proxy, not a future label)
   trend_direction=='down': 16,262 pages  (54.2%)

2) Demand vs a single stale rule
   declining AND impressions_90d>=100: 13,152 pages, 51.2% of impressions
   stale visible (update>=180d AND impressions_90d>=500): 17 pages, 0.1% of impressions

3) Visible low-CTR pages (ctr is a ×100 percentage: 0.5 means 0.5%)
   impressions_90d>=500, position 1-20, ctr<0.5: 9,759 pages, 58.3% of impressions

Data gotcha check (not a lane metric): avg_position==0 means no data, not rank zero
   rows with avg_position==0: 1,205


## 4. Careful words: what I can and can't claim

**I can claim (if the later notebooks earn it).**
- **Observed** patterns on this snapshot: which pages currently show high impressions with weak CTR, freshness, or movement.
- **Directional** ranking quality: whether a score orders review candidates better than a transparent rule, measured by precision@K on a held-out group of clients (and later a time split).
- **Decision-support:** a queue a human can inspect, with reason codes, not an automatic refresh.

**I cannot claim.**
- That refreshing a page **caused** recovery. This data has no experiment.
- **Google ranking factors**, algorithm changes, or "predicting Google."
- That `is_declining_label` / `trend_direction == "down"` is a future outcome. In the starter file it is computed from the **same** 90-day window (`trend_pct` / last-30 vs prev-30 impressions). Using `trend_direction` or `trend_pct` as a feature to predict that label is circular.
- Client names, domains, URLs, or raw queries. Those are not in this file and must not appear in outputs.

**Language I will keep using:** observed, measured, associated with, directional, decision-support, proxy label. I will not write "proves," "will recover if we rewrite," or "Google rewards X."

The cell below checks the label trap on this file: `trend_direction` is the source of the starter decline bucket.

In [14]:
# Label trap: the starter "decline" bucket is defined from trend_direction, not a later window.
print("trend_direction counts:")
print(df["trend_direction"].value_counts().to_string())
print()
proxy = df["trend_direction"] == "down"
print(f"proxy decline rate: {proxy.mean():.1%}  ({int(proxy.sum()):,} / {len(df):,})")
print("This bucket is useful as a baseline/reason code. It is not a future-observed label.")
print("Features that must not train a 'discovery' model of this bucket: trend_direction, trend_pct.")


trend_direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

proxy decline rate: 54.2%  (16,262 / 30,000)
This bucket is useful as a baseline/reason code. It is not a future-observed label.
Features that must not train a 'discovery' model of this bucket: trend_direction, trend_pct.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.